# SFT on HumanEval (Qwen2.5-Coder-3B, LoRA) – HE_V5 (Pinned, FedAvg-ready)

Supervised fine-tuning of **Qwen2.5-Coder-3B-Instruct** on HumanEval using **LoRA** (no quantization), aligned with DS1000/MBPP notebooks.

**V5:** Training target uses **generated_code** for rows where `status == "passed"`, and **canonical_solution** for failed/hallucinated rows.

Input: `FED/HumanEval/human_eval_sft_train.csv` (must have `status` and `generated_code` columns)  
Outputs: PEFT adapter folder + `lora_state_dict.pt` for FedAvg.

In [1]:
# Check GPU (optional; skip if no NVIDIA GPU)
import subprocess
try:
    subprocess.run(["nvidia-smi"], check=True)
except Exception as e:
    print("nvidia-smi not available:", e)

Tue Mar 17 08:48:43 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.288.01             Driver Version: 535.288.01   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA RTX A4000               Off | 00000000:55:00.0 Off |                  Off |
| 41%   44C    P0              45W / 140W |     77MiB / 16376MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [2]:
# Pinned stack to avoid torchao/PEFT key mismatches across save/load.
# If you are using a different CUDA build, adjust only the +cuXXX wheels consistently across torch/vision/audio.
!pip install --no-cache-dir \
  "torch==2.10.0" "torchvision==0.25.0" "torchaudio==2.10.0" \
  "torchao==0.16.0" \
  "transformers==5.2.0" "peft==0.18.1" "trl==0.19.1" \
  "datasets==4.6.1" "accelerate==1.12.0" "pandas==2.3.3" "safetensors"

Defaulting to user installation because normal site-packages is not writeable


## Data loading

In [3]:
import os
import pandas as pd

try:
    NOTEBOOK_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    NOTEBOOK_DIR = os.getcwd()

# Notebook is in FED/HumanEval/HE_V4; CSV is in FED/HumanEval/
CSV_PATH = os.path.abspath(os.path.join(NOTEBOOK_DIR, "human_eval_sft_train.csv"))
print("Using CSV:", CSV_PATH)

df = pd.read_csv(CSV_PATH)
print("Total rows:", len(df))
print("Columns:", list(df.columns))

Using CSV: /home/jovyan/FED/HumanEval/human_eval_sft_ready.csv
Total rows: 164
Columns: ['dataset', 'task_id', 'prompt', 'canonical_solution', 'status', 'generated_code']


In [4]:
# Keep only valid rows (V5 requires status and generated_code for passed/canonical target logic)
required_cols = {"prompt", "canonical_solution", "status", "generated_code"}
missing = sorted(required_cols - set(df.columns))
if missing:
    raise ValueError(f"CSV missing required columns for V5: {missing}")

df_he = df.copy()
if "dataset" in df_he.columns:
    # HumanEval-only (safe even if CSV already contains only HumanEval)
    df_he = df_he[df_he["dataset"].astype(str).str.lower().eq("humaneval")].copy()

df_he = df_he.dropna(subset=["prompt", "canonical_solution"]).copy()
df_he["canonical_solution"] = df_he["canonical_solution"].astype(str).str.strip()
df_he["generated_code"] = df_he["generated_code"].astype(str).str.strip()
df_he = df_he[df_he["canonical_solution"].str.len() > 0]
print("HumanEval rows:", len(df_he))

HumanEval rows: 164


## Parse prompt and build chat messages

In [5]:
SEP = "\n\nUser: "

def row_to_messages(row):
    prompt_str = str(row["prompt"]).strip()
    idx = prompt_str.find(SEP)
    if idx == -1:
        import warnings
        warnings.warn(
            f"Row {row.get('task_id')}: no '\\n\\nUser: ' found; using whole prompt as user.",
            stacklevel=2,
        )
        system_content = "You are an expert Python developer. Complete the function provided by the user."
        user_content = prompt_str.replace("System: ", "", 1).strip()
    else:
        system_content = prompt_str[:idx].replace("System: ", "", 1).strip()
        user_content = prompt_str[idx + len(SEP):].strip()
    # V5: use generated_code when passed, else canonical_solution
    raw_status = str(row.get("status", "")).strip().lower()
    gen_code = str(row.get("generated_code", "")).strip() if "generated_code" in row else ""
    if raw_status == "passed" and len(gen_code) > 0:
        solution = gen_code
    else:
        solution = str(row["canonical_solution"]).strip()
    return [
        {"role": "system", "content": system_content},
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": solution},
    ]

messages_list = [row_to_messages(row) for _, row in df_he.iterrows()]
n_passed = (df_he["status"].astype(str).str.strip().str.lower() == "passed").sum()
n_canonical = len(df_he) - n_passed
print("Built", len(messages_list), "message lists.")
print("Training target: generated_code (passed):", n_passed, "| canonical_solution (failed/hallucinated):", n_canonical)

Built 164 message lists.


/tmp/ipykernel_195/2356310437.py:24: UserWarning: Row HumanEval/0: no '\n\nUser: ' found; using whole prompt as user.
  messages_list = [row_to_messages(row) for _, row in df_he.iterrows()]
/tmp/ipykernel_195/2356310437.py:24: UserWarning: Row HumanEval/1: no '\n\nUser: ' found; using whole prompt as user.
  messages_list = [row_to_messages(row) for _, row in df_he.iterrows()]
/tmp/ipykernel_195/2356310437.py:24: UserWarning: Row HumanEval/2: no '\n\nUser: ' found; using whole prompt as user.
  messages_list = [row_to_messages(row) for _, row in df_he.iterrows()]
/tmp/ipykernel_195/2356310437.py:24: UserWarning: Row HumanEval/3: no '\n\nUser: ' found; using whole prompt as user.
  messages_list = [row_to_messages(row) for _, row in df_he.iterrows()]
/tmp/ipykernel_195/2356310437.py:24: UserWarning: Row HumanEval/4: no '\n\nUser: ' found; using whole prompt as user.
  messages_list = [row_to_messages(row) for _, row in df_he.iterrows()]
/tmp/ipykernel_195/2356310437.py:24: UserWarning: 

In [6]:
from datasets import Dataset

train_dataset = Dataset.from_dict({"messages": messages_list})
print(train_dataset)

Dataset({
    features: ['messages'],
    num_rows: 164
})


## Model and tokenizer

In [7]:
from transformers import AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-Coder-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("Tokenizer loaded.")

Tokenizer loaded.


In [8]:
import torch
from transformers import AutoModelForCausalLM

compute_dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=compute_dtype,
    device_map="auto",
    trust_remote_code=True,
)
print("Model loaded (fp16/bf16, no quantization).")

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Model loaded (fp16/bf16, no quantization).


## LoRA (PEFT) configuration

**FedAvg**: Keep this config identical across clients so keys/shapes match.

In [9]:
from peft import LoraConfig, get_peft_model

LORA_R = 16
LORA_ALPHA = 32
TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=TARGET_MODULES,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


## Training

In [10]:
from trl import SFTTrainer, SFTConfig

OUTPUT_BASE = NOTEBOOK_DIR if "NOTEBOOK_DIR" in dir() else os.getcwd()
output_dir = os.path.join(OUTPUT_BASE, "sft_humaneval_output")
ADAPTER_DIR = os.path.join(output_dir, "lora_adapters")

# Completion-only loss: mask prompt tokens
try:
    from trl import DataCollatorForCompletionOnlyLM
    response_template = "<|im_start|>assistant\n"
    collator = DataCollatorForCompletionOnlyLM(response_template, tokenizer=tokenizer)
except ImportError:
    try:
        from trl.extras import DataCollatorForCompletionOnlyLM
        response_template = "<|im_start|>assistant\n"
        collator = DataCollatorForCompletionOnlyLM(response_template, tokenizer=tokenizer)
    except ImportError:
        collator = None

training_args = SFTConfig(
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=5,
    save_strategy="epoch",
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    max_grad_norm=0.3,
    save_total_limit=1,
    max_seq_length=2048,
    packing=False,
)

def formatting_func(example):
    return tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )

trainer_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    formatting_func=formatting_func,
    processing_class=tokenizer,
)
if collator is not None:
    trainer_kwargs["data_collator"] = collator

trainer = SFTTrainer(**trainer_kwargs)
print("SFTTrainer created.")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Applying formatting function to train dataset:   0%|          | 0/164 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/164 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/164 [00:00<?, ? examples/s]

SFTTrainer created.


In [ ]:
# Train
trainer.train()

## Save adapter (PEFT format)

In [ ]:
os.makedirs(ADAPTER_DIR, exist_ok=True)
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("Saved adapter and tokenizer to", ADAPTER_DIR)
print("Files:", os.listdir(ADAPTER_DIR))

## Save FedAvg-ready `lora_state_dict.pt`

In [ ]:
lora_state = {
    k: v.detach().cpu().clone()
    for k, v in model.named_parameters()
    if v.requires_grad
}

lora_pt_path = os.path.join(ADAPTER_DIR, "lora_state_dict.pt")
torch.save(lora_state, lora_pt_path)
print("Saved FedAvg-ready LoRA state dict to", lora_pt_path)
print("Num tensors:", len(lora_state))
print("File size (MB):", os.path.getsize(lora_pt_path) / (1024 * 1024))